# 01. Data Understanding & Target Leakage Analysis
## VLSI & Hardware: Multi-Class Fault Diagnosis and Mitigation Prediction

**Author:** Senior Data Science & VLSI AI Team  
**Evaluation:** College Course Project (10 Marks Evaluation)  

---

### 1. Project Background & Problem Statement
Modern deep sub-micron (DSM) integrated circuits operating across 7nm to 45nm technology nodes experience physical, electro-thermal, and aging stresses. Automated identification of the root cause of functional degradation is essential for real-time chip self-healing and yield optimization.

The primary objective is to classify circuit operating states into six categories:
1. **No Fault (Nominal)**
2. **Thermal Fault**
3. **Timing Fault**
4. **Power Fault**
5. **Signal Integrity Fault**
6. **Reliability Fault**


In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
from data_loader import load_raw_data, inspect_data, FAULT_CLASS_MAP, BASE_NUMERICAL_FEATURES

# Load raw dataset
df = load_raw_data("../Data/vlsi_fault_detection.csv")
print("Dataset Shape:", df.shape)
df.head(5)


### 2. Dataset Dimensions, Missing Values, and Data Types
We inspect the dataset for data cleanliness, missing values, duplicates, and column types.


In [ ]:
info = inspect_data(df)
print(f"Total Rows: {info['n_rows']:,}")
print(f"Total Columns: {info['n_cols']}")
print(f"Missing Values: {info['missing_values']}")
print(f"Duplicate Rows: {info['duplicates']}")
print("\nTarget Fault Class Distribution:")
for k, v in info['fault_type_counts'].items():
    print(f"  {k} ({FAULT_CLASS_MAP[k]}): {v:,} samples ({v/len(df)*100:.1f}%)")


### 3. Critical Discovery: Deterministic Mapping & Data Leakage Prevention
Let us examine the secondary target column `mitigation_action` and `severity`.


In [ ]:
# Check crosstab between (fault_type, severity) and mitigation_action
crosstab = pd.crosstab([df['fault_type'], df['severity']], df['mitigation_action'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
crosstab


### 4. Summary of Data Understanding Findings
1. **Exact Class Balance**: Exactly 5,000 samples exist for each of the six fault classes (total 30,000 samples).
2. **Zero Missing & Zero Duplicates**: The dataset is completely clean with no null values or duplicate records.
3. **Target Leakage Identified**: `severity`, `mitigation_action`, and one-hot encoded `action_*` dummy variables must be strictly excluded from the feature matrix used for primary fault diagnosis.
4. **Secondary Task Nuance**: Each `(fault_type, severity)` tuple maps 100% deterministically to a specific mitigation action. Therefore, mitigation prediction should be framed as predicting the action directly from raw sensor measurements without knowledge of severity.
